In [2]:
# Import libraries
from dotenv import load_dotenv
import os
from openai import OpenAI
import gradio as gr

c:\Users\User\CS529 Projects\llm-and-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load environment variables
load_dotenv()

True

In [4]:
# Create OpenAI client
client = OpenAI()

In [5]:
# -------------------------------
# SYSTEM PROMPT (VERY IMPORTANT)
# -------------------------------
system_prompt = """
You are a professional Customer Support Assistant for an online store.

Your responsibilities:
- Help customers with orders, refunds, payments, shipping, and product issues
- Be polite, helpful, and concise
- Always sound human and friendly
- If you don't know something, say: "Let me connect you to a human agent."

Guidelines:
- Keep responses short and clear
- Offer solutions whenever possible
- Ask follow-up questions if needed
"""

In [16]:
# -------------------------------
# CHAT FUNCTION
# -------------------------------
def customer_support_chat(message, history):
    if not message or not str(message).strip():
        return "⚠️ Please enter your question."

    def _to_text(content):
        if content is None:
            return ""
        if isinstance(content, str):
            return content
        if isinstance(content, dict):
            return str(content.get("text") or content.get("value") or "")
        if isinstance(content, list):
            parts = []
            for item in content:
                if isinstance(item, str):
                    parts.append(item)
                elif isinstance(item, dict):
                    txt = item.get("text") or item.get("value")
                    if txt:
                        parts.append(str(txt))
            return "\n".join(parts)
        return str(content)

    try:
        convo = [
            {
                "role": "system",
                "content": [{"type": "input_text", "text": system_prompt}],
            }
        ]

        for h in history or []:
            if not isinstance(h, dict):
                continue
            role = h.get("role")
            if role not in ("user", "assistant"):
                continue

            text = _to_text(h.get("content")).strip()
            if not text:
                continue

            # Responses API requires different content types by role
            ctype = "input_text" if role == "user" else "output_text"
            convo.append(
                {
                    "role": role,
                    "content": [{"type": ctype, "text": text}],
                }
            )

        convo.append(
            {
                "role": "user",
                "content": [{"type": "input_text", "text": str(message).strip()}],
            }
        )

        response = client.responses.create(
            model="gpt-5-mini",
            input=convo,
        )

        return response.output_text or "I could not generate a response."

    except Exception as e:
        return f"❌ Error: {e}"

In [19]:
# -------------------------------
# GRADIO UI
# -------------------------------
demo = gr.ChatInterface(
    fn=customer_support_chat,
    title="🛍️ Customer Support Chatbot",
    description="Ask about orders, refunds, shipping, or products."
)

demo.launch(share=True, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 08:57:56 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-04-01T08:57:56-05:00 is after 2026-04-01T07:01:35Z
